# X-ray emission example

This example computes the absorbed spectrum for three different mock neutron stars with different X-ray luminosities and magnetic field values.

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import pathlib
import pickle
import scipy.integrate as integrate
import scipy.special as scsp

from scipy.integrate import trapz, quad
import pypopsyn.simulator.basics.constants as const
import pypopsyn.simulator.multiband_emission.emission_xray as xem
import pypopsyn.simulator.interstellar_medium.nh_model as nhm
import pypopsyn.simulator.interstellar_medium.xray_abs_cross_section as xabs
import utilities.plot_settings
from pypopsyn.simulator.config_simulator import cfg

In [ ]:
# Load the interpolator function to evaluate the x-ray luminosity.
interpolator_Lx_path = pathlib.Path().joinpath(
    cfg["path_to_software"],
    "pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/interpolator_Lx.pkl",
)
with open(interpolator_Lx_path, "rb") as f:
    Lx_interpolator = pickle.load(f)

In [ ]:
# Define some values for the age, initial magnetic field and sky position of the stars.
age = np.array([1.e4, 1.e4, 1.e4])
B = np.array([5.e12, 5.e13, 5.e14])
RA = np.array([60.0, 120.0, 250.0])
DEC = np.array([35.0, 45.0, 55.0])
d = np.array([5.0, 10.0, 15.0])
d_cm = d * const.KPC_TO_CM

# Apply the interpolator to find the corresponding X-ray luminosity.
Lx = Lx_interpolator.ev(age, B)

# Compute the observed black-body surface temperature and radius.
T_obs = xem.T_from_Lx(Lx)
R_obs = cfg["NS_radius"] / xem.gr_correction

# Evaluate the corresponding optical depth and thermal velocity of the magnetospheric plasma.
tau_res = xem.resonant_optical_depth(B)
tau_0 = tau_res / 2.0
beta_T = xem.beta_plasma(B)

# Define an array of energies and convert it from [eV] to [erg].
E = np.logspace(1.0, np.log10(20000), 1000)
E_erg = E * const.EV_TO_ERG

In [ ]:
# Compute the black-body intensity spectrum and convert it into [photons cm^-2 s^-1 erg^-1 sterad^-1].
I_bb = xem.blackbody_intensity_spectrum(E_erg, T_obs)
I_ph_bb = I_bb / (E_erg)

# Compute the RCS spectrum and convert it into [erg cm^-2 s^-1 erg^-1 sterad^-1].
I_rcs = (
    xem.resonant_cyclotron_scat_spectrum(
        E_erg, E_erg, tau_0, beta_T, I_ph_bb, n_reflections=4
    )
    * E_erg
)

# Convert the intensities from [erg cm^-2 s^-1 erg^-1 sterad^-1] to [erg cm^-2 s^-1 eV^-1 sterad^-1].
I_bb = I_bb * const.EV_TO_ERG
I_rcs = I_rcs * const.EV_TO_ERG

# Estimate the N_H column density.
N_H = nhm.compute_NH(RA, DEC, d)
# Reshape N_H to make it compatible for broadcasting.
N_H = N_H[:, np.newaxis]

# Estimate the X-ray absorption cross section.
sigma_ISM = xabs.absorption_cross_section_tot(E, cfg["ISM_abundances"])

# Compute the absorbed intensity spectrum.
absorb_factor = np.exp(-sigma_ISM * N_H)
I_absorbed = absorb_factor * I_rcs

In [ ]:
# Convert intensities into fluxes as observed on Earth.
d_cm = d_cm[:, np.newaxis]
flux_bb = (R_obs / d_cm) ** 2 * np.pi * I_bb
flux_rcs = (R_obs / d_cm) ** 2 * np.pi * I_rcs
flux_absorbed = (R_obs / d_cm) ** 2 * np.pi * I_absorbed

In [ ]:
# Index to select a specific neutron star, choose between 0 and 2.
index = 0

fig, ax = plt.subplots(figsize=(12, 10))

# ax.set_xscale('log')
# ax.set_yscale('log')
ax.set_xlim(0.1, 10.0)
# ax.set_ylim(1.e-1,1.e20)
ax.set_xlabel(r"Energy [keV]")
ax.set_ylabel(r"$S_{\rm X}(E)$ [erg cm$^{-2}$ s$^{-1}$ eV$^{-1}$]")

ax.plot(
    E * 1.0e-3,
    flux_bb[index, :],
    linestyle="--",
    linewidth=4,
    color="black",
    rasterized=True,
    label=r"BB intensity",
)
ax.plot(
    E * 1.0e-3,
    flux_rcs[index, :],
    linestyle="-",
    linewidth=4,
    color="tab:orange",
    alpha=1,
    rasterized=True,
    label=r"RCS spectrum",
)
ax.plot(
    E * 1.0e-3,
    flux_absorbed[index, :],
    linestyle="-",
    linewidth=4,
    color="tab:red",
    alpha=1,
    rasterized=True,
    label=r"Absorbed RCS spectrum",
)
plt.legend(frameon=False, loc=0)
plt.grid()

In [ ]:
# Compute the total absorbed flux in the energy band [0.01, 10] keV by integrating the absorbed RCS spectrum in energy.
flux_abs_bolometric = xem.flux_xray_absorbed(Lx, B, RA, DEC, d)
print(flux_abs_bolometric)